# ALCF Job Submission via the AmSC Python Client

This notebook demonstrates how to submit and manage compute jobs on ALCF resources
(Polaris, Aurora, Sophia, etc.) using the AmSC Python Client's facility integration.

**What you'll do:**
1. Create an AmSC client
2. Connect to the ALCF facility and explore available compute resources
3. Submit a "hello world" job to Polaris
4. Monitor the job until completion

**Prerequisites:**
- `amsc-client` installed (with `globus-sdk`)
- A valid ALCF account and project allocation
- A Globus account linked to your ALCF identity
- **IRI API allowlist access** — having an ALCF account is not sufficient. Email [ALCF support](https://help.alcf.anl.gov) with your ALCF username and use case to be added to the API access list. Without this, job submission returns HTTP 401.

**Authentication:**
- ALCF facility endpoints (resource listing, incidents) are **public** — no login needed
- Job submission requires a **Globus login** — a browser window will open automatically

In [ ]:
import os
import time
import amsc_client
print(amsc_client.version_info())
from amsc_client import Client, Resource, Job, ApiError

## Step 1: Create the AmSC Client

The client manages authentication and provides access to all AmSC services.
For this tutorial, we only need facility access — no catalog auth is required.

In [ ]:
# Create a minimal client — we only need facility access, not catalog
client = Client(token="not-needed-for-facilities")
print("✅ Client created")

# Included catalog config in case you would like to expand and play with this notebook:
# base_url='https://api.american-science-cloud.org/api/current'
# globus_app_id='e4f48665-38b5-4833-a89e-849c71f5b3e3' # ChildersAmSC ->  AmSC CLI Client
# client = Client(
#     base_url=base_url,
#     auth_method="globus",
#     globus_client_id=globus_app_id,
#     requested_scopes=f'openid profile email https://auth.globus.org/scopes/{globus_app_id}/amsc_test',
#     resource_server=globus_app_id,
#     use_id_token=True,
# )

## Step 2: Connect to ALCF and Explore Resources

ALCF is a built-in facility — just call `client.facility("alcf")`.
Resource listing is public (no auth required).

In [ ]:
# Connect to ALCF
alcf = client.facility("alcf")

# Get facility info
info = alcf.info()
print(f"Facility: {info.name}")
print(f"Organization: {getattr(info, 'organization_name', 'N/A')}")

In [ ]:
# List all available resources
resources = alcf.resources()

print(f"Available Resources ({len(resources)}):\n")
print(f"{'Name':12s}  {'Type':10s}  {'Status'}")
print(f"{'─'*12}  {'─'*10}  {'─'*12}")
for r in resources:
    print(f"{r.name:12s}  {r.resource_type:10s}  {r.status}")

In [ ]:
# Get a specific compute resource by name (case-insensitive)
polaris = alcf.resource("Polaris")

print(f"Resource: {polaris.name}")
print(f"ID:       {polaris.id}")
print(f"Type:     {polaris.resource_type}")
print(f"Status:   {polaris.status}")

## Step 3: Check for Active Incidents

Before submitting a job, it's good practice to check for any active
outages or maintenance windows.

In [ ]:
incidents = alcf.incidents()
print(f"Total incidents: {len(incidents)}")

# Show the 5 most recent
if incidents:
    print(f"\nRecent incidents:")
    for inc in incidents[:5]:
        print( f' - {str(inc.last_modified):20} | {inc.name:25} -> {inc.resolution:20} ')

## Step 4: Configure and Submit a Job

Now we'll submit a simple "hello world" job to Polaris.

**⚠️ This will open a browser window for Globus login** (ALCF credentials).
You need to authenticate with your ALCF account to submit jobs.

**Before running:** Update `ALCF_ACCOUNT` below to match your ALCF project allocation.

In [ ]:
# ── Job Configuration ──────────────────────────────────────────────
# Update these to match your ALCF account:

ALCF_ACCOUNT  = "datascience"                      # Your ALCF project allocation
ALCF_USERNAME = "parton"                          # Your ALCF username
ALCF_QUEUE    = "debug"                           # Queue (debug for quick tests)
OUTPUT_DIR    = f"/home/{ALCF_USERNAME}/iri_job_outputs"  # Must exist on the filesystem

# ─────────────────────────────────────────────────────────────────

import datetime
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
JOB_NAME = f"amsc-tutorial-{RUN_ID}"

print(f"Job name:    {JOB_NAME}")
print(f"Resource:    {polaris.name}")
print(f"Queue:       {ALCF_QUEUE}")
print(f"Account:     {ALCF_ACCOUNT}")
print(f"Output dir:  {OUTPUT_DIR}")

In [ ]:
# Submit the job
# This triggers Globus login on first call (browser opens)

job = polaris.submit(
    executable="/bin/echo",
    arguments=["Hello from AmSC Python Client!", RUN_ID],
    directory=OUTPUT_DIR,
    name=JOB_NAME,
    queue=ALCF_QUEUE,
    account=ALCF_ACCOUNT,
    duration=300,        # Wall time in seconds (5 minutes)
    nodes=1,             # Number of nodes
    filesystems="home",  # Filesystems to mount
)

print(f"\n✅ Job submitted!")
print(f"   Job ID:    {job.id}")
print(f"   State:     {job.state}")
print(f"   {job!r}")

## Step 5: Monitor the Job

The `Job` object is "self-aware" — it knows which facility and resource it belongs to,
and can refresh its own status from the API.

We can poll manually or use `job.wait()` to block until completion.

In [ ]:
# Manual polling — check status every 5 seconds

POLL_TIMEOUT = 60  # seconds
POLL_INTERVAL = 5  # seconds

print(f"Polling job {job.id} (up to {POLL_TIMEOUT}s)...\n")
start_time = time.time()

while time.time() - start_time < POLL_TIMEOUT:
    elapsed = int(time.time() - start_time)
    try:
        current_status = job.status  # calls the API
        state = job.state
        print(f"  [{elapsed:3d}s] State: {state}")

        if job.is_terminal:
            exit_code = job.exit_code
            message = job.message
            print(f"\n✅ Job finished!")
            print(f"   Final state: {state}")
            if exit_code is not None:
                print(f"   Exit code:   {exit_code}")
            if message:
                print(f"   Message:     {message}")
            break
    except Exception as e:
        # Completed PBS jobs disappear from the queue;
        # the API returns 400 "not found" in that case
        if "not found" in str(e).lower() or "400" in str(e):
            print(f"  [{elapsed:3d}s] Job no longer in scheduler queue (likely completed)")
            print(f"\n✅ Job completed (exited scheduler)")
            break
        else:
            print(f"  [{elapsed:3d}s] Status check error: {e}")

    time.sleep(POLL_INTERVAL)
else:
    print(f"\n⏰ Timed out after {POLL_TIMEOUT}s (job may still be running)")
    print(f"   Last known state: {job.state}")

## Step 6: Read Job Output via the Filesystem API

Now that the job has completed, we can read its output directly
using the **filesystem API** — no need to SSH into the machine.

We use **Home** (ALCF's home filesystem resource) to access the output files,
since job output was written to the user's home directory.

In [ ]:
# Access the filesystem through the Home resource
home = alcf.resource("Home")

stdout_path = f"{OUTPUT_DIR}/{JOB_NAME}.stdout"
stderr_path = f"{OUTPUT_DIR}/{JOB_NAME}.stderr"

# Read the job's stdout
try:
    task = home.fs.head(stdout_path, lines=50)
    task.wait(timeout=60)
    print(f"── {stdout_path} ──")
    print(task.result)
except ApiError as e:
    print(f"🚧 head: Not available yet — {e}")

# Read the job's stderr (should be empty for a successful job)
try:
    task = home.fs.head(stderr_path, lines=50)
    task.wait(timeout=60)
    print(f"\n── {stderr_path} ──")
    print(task.result if task.result else "(empty)")
except ApiError as e:
    print(f"🚧 head: Not available yet — {e}")

## Alternative: Using `job.wait()`

Instead of manual polling, you can use the built-in `wait()` method
which blocks until the job reaches a terminal state:

In [ ]:
# Submit and wait for completion

job2 = polaris.submit(
    executable="/bin/hostname",
    directory=OUTPUT_DIR,
    name=f"amsc-wait-demo-{RUN_ID}",
    queue=ALCF_QUEUE,
    account=ALCF_ACCOUNT,
    duration=300,
    nodes=1,
    filesystems="home",
)
print(f"✅ Job submitted: {job2.id}")
print(f"   Waiting for completion...")

try:
    job2.wait(timeout=120, poll_interval=5)
    print(f"✅ Job completed: state={job2.state}, exit_code={job2.exit_code}")
except TimeoutError:
    print(f"⏳ Job still running after timeout (state: {job2.state})")

## Summary

This tutorial showed how to:

| Step | API Call | Auth Required? |
|------|---------|----------------|
| Connect to ALCF | `client.facility("alcf")` | No |
| List resources | `alcf.resources()` | No |
| Get resource details | `alcf.resource("Polaris")` | No |
| Check incidents | `alcf.incidents()` | No |
| Submit a job | `polaris.submit(...)` | Yes (Globus → ALCF) |
| Check job status | `job.status` | Yes |
| Wait for completion | `job.wait(timeout=120)` | Yes |
| Read job output | `eagle.fs.head(path)` | Yes |
| Cancel a job | `job.cancel()` | Yes |

**Key concepts:**
- **Resource-scoped compute**: Jobs are submitted via the resource object (`polaris.submit(...)`) rather than a separate compute client
- **Self-aware jobs**: The `Job` object tracks its own facility and resource, and can refresh status, wait, or cancel itself
- **Filesystem via Eagle**: Use `eagle.fs.head()` to read job output files without SSH — Eagle sees the same home directories as compute nodes
- **Lazy authentication**: Globus login is only triggered when you first call an authenticated endpoint (e.g., `submit()`)
- **PBS behavior**: Completed jobs disappear from the scheduler queue; the API returns a 400 error when this happens

## Troubleshooting

### 401 on Job Submission (Not on IRI API Allowlist)

If you get an `HTTP 401` error when submitting a job (after successfully logging in), your account may not be on the ALCF IRI API access list. Having an ALCF account and allocation is not enough — access to the IRI API must be requested separately. Email [ALCF support](https://help.alcf.anl.gov) with your ALCF username and use case.

### Persistent 401 Errors After Re-Authentication

ALCF tokens embed a Keycloak identity token inside the Globus access token. This embedded token can expire even while the Globus token itself remains valid. If you see repeated `AuthenticationError: Authentication failed (401)` errors even after re-authenticating:

1. **Clear your browser cookies** for `globus.org` and `globusid.org` domains
2. **Delete cached credentials**: `rm ~/.amsc/credentials.json`
3. **Restart your notebook kernel** and re-run from the beginning

This forces a full fresh login through ALCF's Keycloak identity provider, which issues a new embedded identity token.